# Session 5 — Agentic AI I: Generative AI Basics + What Agents Are
**Data Science Techniques and Real-World Applications — WS 2026**

**Zahra Sharafi**

**Frankfurt School of Finance and Management**

Thursday 8 October 2026


In [10]:
import re
import pandas as pd
import numpy as np

## <span style="color:#1e3a8a">Generative AI basics: how a language model works</span>

A **large language model (LLM)** is a model that predicts the next piece of text. To see what that
means in practice, start with the unit it works with: the **token**.

### Step 1: text becomes tokens

A model doesn't read characters or whole words. A **tokenizer** cuts text into **tokens**: often a
whole word, sometimes a fragment, sometimes a space or punctuation mark. Common words are one token;
rare or invented words are split into familiar pieces. Note that the space before a word is usually
part of that word's token (`" revenue"`, not `"revenue"`).

In [11]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")   # a tokenizer used by many OpenAI models
for text in ["Apple announced its new iPhone.", "AAPL", "unbelievably", "Q3 revenue grew 14.7%"]:
    ids = enc.encode(text)
    print(f"{text!r:35} {len(ids)} tokens: {[enc.decode([i]) for i in ids]}")

'Apple announced its new iPhone.'   6 tokens: ['Apple', ' announced', ' its', ' new', ' iPhone', '.']
'AAPL'                              2 tokens: ['AAP', 'L']
'unbelievably'                      3 tokens: ['un', 'belie', 'vably']
'Q3 revenue grew 14.7%'             9 tokens: ['Q', '3', ' revenue', ' grew', ' ', '14', '.', '7', '%']


Read the output: `AAPL` costs two tokens (`AAP` + `L`), because the tokenizer has no single piece for
that exact string. Numbers and symbols like `14.7%` split into several tokens too. This matters
later: cost is counted in tokens, not words.

*Note*: not random, but also not a menu of equally-valid alternatives — one fixed algorithm, one fixed output, every time.

### Step 2: next-token prediction, and temperature

Given the tokens so far, the model assigns a **probability to every token** in its vocabulary (on the
order of 100,000 possibilities), picks one, appends it, and repeats. A full answer is this loop run
hundreds of times.

**Temperature** controls how those probabilities are used when picking. Before picking, every
probability is raised to the power `1/T`, then the results are renormalized to sum to 1:

- **T below 1 (say 0.3)**: sharpens the distribution. The most likely token dominates, so answers
  are more predictable and repetitive.
- **T = 1**: uses the model's probabilities as they are.
- **T above 1 (say 3)**: flattens the distribution. Unlikely tokens get a real chance, so answers are
  more varied, and also more prone to errors.

Temperature doesn't change what the model **knows**; it only changes how it **chooses** from what it
knows. Example: after "The stock went", suppose the model has seen "up" 7 times and "down" 4 times.
The code below applies different temperatures to those counts.

In [12]:
counts = {"up": 7, "down": 4}                     # how often each continuation was seen
words = list(counts.keys())
weights = np.array(list(counts.values()), dtype=float)

for T in [0.3, 1.0, 3.0]:
    sharpened = weights ** (1 / T)                 # raise each weight to the power 1/T
    probs = sharpened / sharpened.sum()            # renormalize so the probabilities sum to 1
    print(f"temperature={T}: {({w: round(float(p), 3) for w, p in zip(words, probs)})}")

temperature=0.3: {'up': 0.866, 'down': 0.134}
temperature=1.0: {'up': 0.636, 'down': 0.364}
temperature=3.0: {'up': 0.546, 'down': 0.454}


Read the output: at **T = 0.3**, "up" gets about 87% of the picks, so the model almost always says
"up". At **T = 1**, the raw 7-to-4 split (64% / 36%) is used. At **T = 3**, the two options are nearly
a coin flip (55% / 45%). Same model, same knowledge, three different behaviors.

For an agent, this is why you usually want a **low** temperature when it's extracting facts or
calling tools (you want the most likely, most literal answer), and a higher one only for brainstorming.

### Attention: which earlier tokens matter for the next one

Inside the model, each token gets to look back at the tokens before it and decide how much each one
matters for predicting what comes next. Concretely: the token's current representation is compared
with every earlier token's representation, the comparison scores are turned into **weights that sum
to 1** (a softmax), and those weights are used to blend the earlier tokens' information into the
current token. The effect is that a token's representation depends on its context. The "Apple" in
"Apple announced its new iPhone" is updated by "iPhone" nearby, so it ends up with a different
representation than the "Apple" in "apple orchard harvest".

Illustration: predicting the token after `Apple announced its new iPhone`. These weights are
**hand-set for illustration**, not measured from a trained model:

| Earlier token attended to | Attention weight (illustrative) |
|---|---|
| `Apple` | 0.30 |
| `announced` | 0.10 |
| `its` | 0.05 |
| `new` | 0.15 |
| `iPhone` | 0.40 |
| **sum** | **1.00** |

Real models run many attention heads in many layers, each learning a different pattern (one might
track grammar, another which noun a pronoun refers to). The mechanism is the same.

### BERT vs. GPT: two training objectives

Both families are built from transformers, which means they use attention. They differ in what they
are trained to predict, and that changes what they are good at:

- **GPT-style (decoder-only, causal)**: trained to predict the **next** token. Each token may look only
  at tokens to its left: a causal mask stops it from peeking at the future. Training example:
  `"Apple announced its new"` &rarr; predict `"iPhone"`. Because it's trained to continue text, it's the
  family used for generation: chatbots, agents, writing answers.
- **BERT-style (encoder-only, bidirectional)**: trained by hiding (masking) random tokens and predicting
  them, using context from **both** sides. Example: `"Apple announced its new [MASK]"`, where the model
  can also use any words after the mask. Because it sees the whole sentence at once, it's built for
  *understanding* text: classification, search, extracting entities. It isn't trained to write
  fluent text one token at a time.
    That makes it great at:
    - Classifying text (spam vs. not spam, positive vs. negative review)
    - Search ("does this document match this query?")
    - Pulling out specific info ("find the company name in this sentence")

Rule of thumb for agents: a BERT-style model is often used to *read and classify* a document; a
GPT-style model writes the answer.

### Three limits, in more detail

#### 1. Hallucination

The model picks the most *probable* continuation of text. Nothing in that process checks a claim
against a source, so a fluent wrong answer and a fluent right answer come out looking identical.
The confidence is in the **wording**, not in the **fact**.

**Example:** ask for "Apple's diluted EPS for 2019," and the model can produce a precise-looking
number reconstructed from patterns in its training text, not from an actual filing.

**What helps:**
- Put the source text in the prompt (RAG, next section).
- Ask for citations, and check them.
- Keep the temperature low for factual tasks.
- Verify numbers against a database before anyone acts on them.

#### 2. Bias

The model's probabilities reflect the text it was trained on. If that text links certain roles,
names, or groups with certain traits, the model reproduces those links as its "most likely"
completions. No rule says to do this; it lives purely in the statistics of the training data.

**Example:** asked to complete "The typical fund manager is ___," a model trained on text where
that role is mostly described as male may prefer male pronouns.

**What helps:**
- Test outputs with matched prompts that differ only in a group attribute.
  - Build two (or more) prompts that are identical except for the one thing you're testing (gender,
    name, ethnicity, ...), and compare what the model says. For example:
    - Prompt A: `"John is a fund manager. Write a short performance review for him."`
    - Prompt B: `"Maria is a fund manager. Write a short performance review for her."`

    Everything else is held constant: same role, same request, same length. If the model
    consistently describes "John" as "decisive" and "aggressive" but "Maria" as "emotional" or
    "collaborative" across many such pairs, that's measurable, reproducible evidence of bias, not a
    guess or a single anecdote.
- Document where the training data came from.
- Keep a human in the loop for any decision about people.
- Never treat raw model output as a verified decision.
  - A model-generated credit approval, hiring recommendation, or performance rating still needs a human to sign off before it affects someone's actual outcome.

#### 3. Cost and latency

Providers bill per token, for input and for output. Everything you send &mdash; system
instructions, retrieved documents, conversation history &mdash; is paid for on every single call.
The model also writes its answer one token at a time, so longer answers take longer.

**Example:** the code below counts the input tokens of a repeated prompt and projects a daily
bill. The price used is an **illustrative placeholder**, not a quoted rate &mdash; check your
provider's current pricing.

**What helps:**
- Keep prompts short and relevant (RAG retrieves only the top passages).
- Cache repeated prompts.
- Use a smaller model for simple steps.
- Cap the output length.

In [13]:
prompt = "Summarize the following excerpt from Apple's annual report. " * 40
n_tokens = len(enc.encode(prompt))
print("input tokens per call:", n_tokens)

calls_per_day = 10_000
price_per_1k_input = 0.003   # ILLUSTRATIVE placeholder, not a real quoted price

daily_cost = n_tokens / 1000 * price_per_1k_input * calls_per_day
print(f"illustrative input cost at {calls_per_day:,} calls/day: ${daily_cost:,.2f}")

input tokens per call: 481
illustrative input cost at 10,000 calls/day: $14.43


### RAG: Retrieval-Augmented Generation

**The idea, in one line: RAG is an open-book exam.** Without it, the model answers purely from
memory (what it absorbed during training) &mdash; frozen, sometimes wrong, and silent about
anything recent or private. With RAG, the model first *looks up the relevant page*, then writes
its answer from that page, the same way you'd flip to the right chapter before answering an exam
question instead of guessing from memory.

**The three steps:**

1. **Retrieve**: search a knowledge base (filings, internal documents, a database) for passages
   relevant to the question &mdash; "find the right page."
2. **Augment**: build the prompt from the retrieved passages, the question, and an instruction
   such as "answer only from this context, and say so if the answer isn't there" &mdash; "open the
   book to that page."
3. **Generate**: the LLM writes the answer, now grounded in text you can show and check &mdash;
   "write the answer, looking at the page."

That's the whole concept. Everything below just shows each of the three steps actually running, one
at a time, on four short company descriptions.

### Step 1: Retrieve &mdash; "find the right page"

By hand, on two one-sentence documents, before any code: say the question is *"which company
streams video?"* and we have `AAPL: "Apple sells iPhones and Macs."` and
`NFLX: "Netflix streams video shows."` The question shares the word "video" with NFLX and shares
nothing with AAPL &mdash; so NFLX is the better match. That's the entire idea: **count how many
important words two pieces of text have in common; more overlap means more likely relevant.**

The code below does exactly that, just at scale and with one refinement: not all words count
equally. Here's each piece of that, one short idea at a time:

- **Document**: a text. Real systems split long documents into chunks of a few hundred words;
  here, each document is one short company description.
- **TF-IDF vector**: each document becomes a list of numbers, one per vocabulary word. Each number
  is **TF &times; IDF**:
  - **TF (term frequency)**: how often that word appears in *this* document.
  - **IDF (inverse document frequency)**: words that appear in *few* documents count more, so
    common filler words like "the" count for almost nothing, and distinctive words like
    "streaming" count for a lot.
  - Stop words ("the," "and," "of," "it") are dropped before any of this, since they appear
    everywhere and carry no information.
- **Query vector**: the question is converted the same way, using the vocabulary and weights
  learned from the documents (not relearned from the question).
- **Cosine similarity**: compares the angle between the question's vector and each document's
  vector. 1 means the same words with the same relative weights; 0 means no shared words at all.
  Documents are ranked by this score and the top `k` are kept.

If the TF-IDF detail doesn't fully stick yet, that's fine &mdash; the "count the overlap" idea
above is the one that matters most; everything else is just a more careful way of counting.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = {
    "AAPL": "Apple designs consumer electronics including the iPhone, iPad, and Mac computers. "
            "It earns most of its revenue from hardware sales and a growing services business "
            "including the App Store.",
    "GOOG": "Alphabet, the parent of Google, dominates internet search and online advertising. "
            "It also owns YouTube and develops the Android mobile operating system and cloud "
            "computing services.",
    "AMZN": "Amazon operates the largest e-commerce marketplace in the United States and runs "
            "Amazon Web Services, a major cloud computing platform used by businesses worldwide.",
    "NFLX": "Netflix is a subscription video streaming service producing and licensing original "
            "films and television series, competing with other streaming platforms for viewers.",
}

vectorizer = TfidfVectorizer(stop_words="english")
doc_vectors = vectorizer.fit_transform(documents.values())
tickers = list(documents.keys())

def retrieve(query, k=2):
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, doc_vectors)[0]
    ranked = sorted(zip(tickers, similarities), key=lambda pair: -pair[1])
    return ranked[:k]

for query in ["internet search and advertising", "e-commerce marketplace", "video streaming service"]:
    print(f"{query!r:35} -> {retrieve(query)}")

'internet search and advertising'   -> [('GOOG', 0.4378195912259568), ('AAPL', 0.0)]
'e-commerce marketplace'            -> [('AMZN', 0.3274678315761284), ('AAPL', 0.0)]
'video streaming service'           -> [('NFLX', 0.5601120336112038), ('AAPL', 0.0)]


Read the output: each question returns the best-matching company and a score. "internet search and
advertising" ranks GOOG first because "internet", "search" and "advertising" all appear in its
description. The second entry, `AAPL 0.0`, means zero shared words: it fills the second slot, it is
not a match. Whatever `retrieve()` returns is exactly what step 2 (Augment) would paste into the
prompt.

**Limit to remember:** TF-IDF only matches exact words. "Smartphone maker" shares no words with the
Apple text, so it would score 0 even though it clearly means Apple. Production RAG systems use
**embeddings** (learned vectors where related meanings sit close together, so "smartphone" and
"iPhone" count as similar even with zero words in common), which fixes this. The three-step
pipeline (Retrieve &rarr; Augment &rarr; Generate) stays exactly the same either way.

### Step 2: Augment &mdash; "open the book to that page"

Retrieval alone isn't useful by itself; the retrieved text has to actually reach the model. This
step just builds that text: take the best-matching document(s), paste them into a template
alongside the question and an instruction to stick to that text.

In [19]:
def build_augmented_prompt(query, k=1):
    top_matches = retrieve(query, k=k)
    context = "\n".join(f"[{ticker}]: {documents[ticker]}" for ticker, _ in top_matches)
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the context doesn't contain the answer, say so.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    return prompt

print(build_augmented_prompt("Which company runs a large cloud computing platform?"))

Answer the question using ONLY the context below. If the context doesn't contain the answer, say so.

Context:
[AMZN]: Amazon operates the largest e-commerce marketplace in the United States and runs Amazon Web Services, a major cloud computing platform used by businesses worldwide.

Question: Which company runs a large cloud computing platform?
Answer:


That whole block of text &mdash; not just the one-line question &mdash; is what actually gets sent
to the model. Notice the instruction at the top ("use ONLY the context... say so if it doesn't
contain the answer"): that single sentence is most of what keeps a RAG system grounded instead of
free-associating from memory.

### Step 3: Generate &mdash; "write the answer, looking at the page"

This last step is simply: send the text above to an LLM, the same next-token-prediction process
from the start of this notebook, and let it write the answer. We don't have an API key to call a
real model in class, so we won't run this step live &mdash; but given that prompt, a model would
read the AMZN passage, see it directly answers the question, and write something like *"Amazon,
through Amazon Web Services, runs a major cloud computing platform."* The retrieval and augmentation
steps did the hard work; generation just has to read and summarize text that's already sitting
right there.

## <span style="color:#1e3a8a">What is an AI agent?</span>

Everything described above &mdash; a model that takes text in and predicts text out &mdash; has the
same shape. It has no way to look anything up on its own, take an action, or try again if its first
answer was wrong. That's a real limitation:

- **Passive**: it only responds when prompted; it never initiates anything.
- **No real-world interaction**: it can't query a database, call an API, or run code by itself.
- **No memory between calls**: each request starts from a blank slate (unless *you* re-send the
  earlier conversation yourself).
- **Frozen knowledge**: whatever it learned during training is all it has &mdash; it can't learn
  today's stock price, because today didn't exist yet when it was trained.

An **AI agent** is the fix: wrap an LLM in a loop that lets it *decide* to call tools, look at
what came back, and decide again &mdash; repeating until it has a real answer, not just a
plausible-sounding guess.

### The agent loop: Thought &rarr; Action &rarr; Observation

This pattern has a name &mdash; **ReAct** ("Reasoning + Acting"), from Yao et al., 2022 &mdash; and
it's the backbone of essentially every agent framework in use today:

1. **Thought**: given the goal and everything observed so far, decide what to do next.
2. **Action**: do it &mdash; call a tool (a function, an API, a database query, a calculator).
3. **Observation**: look at what the tool returned.
4. Repeat from 1, now with that observation added to the context, until there's enough to give a
   final answer.

The important part isn't the fancy name &mdash; it's that the model's "thinking" and "doing" are
**interleaved**, one small step at a time, instead of trying to solve everything in one shot. Below
is a deliberately simple, fully rule-based agent (no API key, no real LLM call) built on our own
`techstocks` data from before, so the loop itself &mdash; not the model doing the "thinking"
&mdash; is what's on display.

In [20]:
techstocks = pd.read_excel("data/techstocks.xlsx", sheet_name="main")
avg_returns = techstocks.groupby("Ticker Symbol")["Returns"].mean().to_dict()
print(avg_returns)
# This dict is the agent's "knowledge" — its perception of the world.

{'AAPL': 0.0018085940444016464, 'AMZN': 0.0015740346860655795, 'GOOG': 0.000869914585737639, 'NFLX': 0.0017325936336246308}


In [17]:
# Two "tools" the agent is allowed to use
def tool_get_avg_return(ticker):
    return avg_returns.get(ticker.upper())

def tool_calculator(expression):
    allowed = set("0123456789.+-*/() ")
    if not set(expression) <= allowed:
        raise ValueError("expression contains disallowed characters")
    return eval(expression, {"__builtins__": {}}, {})

# The agent itself: a (deliberately simple, rule-based) "Thought" step, then Action, then Observation
def simple_agent(query):
    print(f"User: {query}")
    compare_match = re.search(r"compare (\w+) and (\w+)", query, re.IGNORECASE)
    single_match = re.search(r"average return of (\w+)", query, re.IGNORECASE)

    if compare_match:
        t1, t2 = compare_match.group(1).upper(), compare_match.group(2).upper()
        print(f"Thought: this needs two lookups, then a comparison -- {t1}'s average return and {t2}'s.")

        print(f'Action: get_avg_return("{t1}")')
        r1 = tool_get_avg_return(t1)
        print(f"Observation: {r1}")

        print(f'Action: get_avg_return("{t2}")')
        r2 = tool_get_avg_return(t2)
        print(f"Observation: {r2}")

        print("Thought: now compute the difference.")
        print(f'Action: calculator("{r1} - {r2}")')
        diff = tool_calculator(f"{r1} - {r2}")
        print(f"Observation: {diff}")

        winner = t1 if r1 > r2 else t2
        print(f"Final answer: {winner} had the higher average daily return "
              f"({max(r1, r2):.5f} vs {min(r1, r2):.5f}, a gap of {abs(diff) * 100:.3f} percentage points).")

    elif single_match:
        t = single_match.group(1).upper()
        print(f"Thought: this is a single lookup for {t}.")
        print(f'Action: get_avg_return("{t}")')
        r = tool_get_avg_return(t)
        print(f"Observation: {r}")
        if r is None:
            print(f"Final answer: I don't have data for {t}.")
        else:
            print(f"Final answer: {t}'s average daily return is {r:.5f} ({r * 100:.3f}%).")

    else:
        print("Final answer: sorry, I can only answer average-return or comparison questions right now.")
    print()

simple_agent("What is the average return of AAPL?")
simple_agent("Compare AAPL and GOOG")
simple_agent("What is the weather today?")

User: What is the average return of AAPL?
Thought: this is a single lookup for AAPL.
Action: get_avg_return("AAPL")
Observation: 0.0018085940444016464
Final answer: AAPL's average daily return is 0.00181 (0.181%).

User: Compare AAPL and GOOG
Thought: this needs two lookups, then a comparison -- AAPL's average return and GOOG's.
Action: get_avg_return("AAPL")
Observation: 0.0018085940444016464
Action: get_avg_return("GOOG")
Observation: 0.000869914585737639
Thought: now compute the difference.
Action: calculator("0.0018085940444016464 - 0.000869914585737639")
Observation: 0.0009386794586640074
Final answer: AAPL had the higher average daily return (0.00181 vs 0.00087, a gap of 0.094 percentage points).

User: What is the weather today?
Final answer: sorry, I can only answer average-return or comparison questions right now.



Notice what's *not* in `simple_agent()`: no internet access, no file access, nothing beyond the
two narrow tools defined above it. That's on purpose. When the third question ("What is the
weather today?") comes in, the agent has no tool for that, so it says so instead of inventing an
answer &mdash; a direct consequence of only being given tools for what it's actually meant to do.

Walk through the second call (`"Compare AAPL and GOOG"`) line by line: the agent didn't answer in
one shot. It recognized it needed *two* pieces of information it didn't have, fetched them one at
a time (two separate Action/Observation pairs), *then* reasoned about what to do with them
(subtract), *then* answered. A plain LLM call, with no tools, would have had to either refuse or
guess at both numbers from memory &mdash; and might have guessed wrong without telling you.

Real agents replace the regex "Thought" step with an actual LLM call (the model reads the query
and the conversation so far, and decides in natural language what to do next) &mdash; but the loop
structure, and the reason it helps, is identical to what you just watched run.

### Four pieces every agent has

Strip away the branding and every agent &mdash; ours included &mdash; is built from the same four
pieces:

| Component | What it does | In our toy agent |
|---|---|---|
| **Reasoning engine** | Decides what to do next (the "Thought" step) | The `re.search()` pattern matching &mdash; a real agent uses an LLM here instead |
| **Perception** | Reads in whatever the current situation/data is | `avg_returns`, loaded once from `techstocks.xlsx` |
| **Action system** | Executes the chosen step | `tool_get_avg_return()`, `tool_calculator()` |
| **Memory** | Carries information across steps (and, in real agents, across conversations) | Just the local variables `r1`, `r2` within one call &mdash; our agent forgets everything the moment `simple_agent()` returns |

That last row is worth dwelling on: our agent has essentially **no memory** beyond a single call.
A real agent typically keeps a **short-term memory** (the running conversation/tool-call history
within one task) and sometimes a **long-term memory** (facts or preferences that persist across
separate sessions, often stored in a database or vector store). Memory is also exactly what turns
a one-off tool-using script into something that *feels* like it's "learning" as a conversation
goes on &mdash; even though, under the hood, it's just re-reading its own history each time.

### Agent vs. chatbot, in one line

A chatbot is **request &rarr; response**: it answers what you asked, once, from what it already
knows. 

An agent is **goal &rarr; loop of actions**: it can take several steps, use tools, and only
answer once it has actually gathered what it needs &mdash; as our AAPL-vs-GOOG example just did.
Not every task needs an agent; a single well-crafted prompt is faster, cheaper, and easier to
predict when a chatbot-style answer is all the task requires.

## <span style="color:#1e3a8a">Permissions & verification</span>

Giving a model the ability to *act*, not just talk, raises the stakes considerably. A chatbot that
hallucinates gives you a wrong sentence; an agent that hallucinates can run the wrong command, send
the wrong email, or spend real money. The hallucination point from the basics above is no longer
just an embarrassing wrong answer, it's now a wrong *action*.

**A wrong action is worse than a wrong sentence.**

Two guardrails matter most in practice:

- **Restrict what each tool can actually do.** Give the agent the narrowest tool that does the job
  &mdash; a tool that can *only* look up a stock return can't also delete a file, no matter what
  the model's "Thought" step decides to try.
- **Verify before anything consequential happens.** For read-only actions (like our lookups above)
  that's usually unnecessary. For anything that writes, sends, spends, or deletes, a human
  confirmation step before execution is the norm, not the exception.

Our calculator tool already does the first one &mdash; it only allows digits and `+-*/()`,
nothing else. Here's why that restriction matters:

In [18]:
# A naive calculator tool: whatever text it's handed, it hands straight to eval().
def tool_calculator_unsafe(expression):
    return eval(expression)

# Our actual tool, which only allows arithmetic characters through.
def tool_calculator_safe(expression):
    allowed = set("0123456789.+-*/() ")
    if not set(expression) <= allowed:
        raise ValueError("expression contains disallowed characters")
    return eval(expression, {"__builtins__": {}}, {})

# A model doesn't need malicious intent for this to matter -- it just needs to pass through
# a string it generated itself that happens to look "helpful," like reading the server's
# own working directory to "check a file path."
sneaky_input = "__import__('os').getcwd()"

print("unsafe tool runs it:   ", tool_calculator_unsafe(sneaky_input))
try:
    tool_calculator_safe(sneaky_input)
except ValueError as e:
    print("restricted tool blocks it:", e)

unsafe tool runs it:    c:\Users\sharafi\Dropbox\ML course\Data Analytics Course\Data Science Course WS2026\Session 5
restricted tool blocks it: expression contains disallowed characters


`eval()` will run *any* Python it's given &mdash; our unsafe version treated the agent's own
generated text as code and executed it without question. This is a toy, deliberately tame example
(`getcwd()` is harmless), but the pattern generalizes: never let a model's free-form text reach a
tool that can do something you wouldn't want to happen on its own, confident or not. The same logic
scales up directly: an agent with file-system access should only touch an allow-listed folder; an
agent with email access should draft, not send, without a human clicking "yes."

None of this requires the agent to be malicious &mdash; it just does whatever its tools allow,
confidently. The restriction is what keeps it safe, not good intentions it doesn't actually have.

## <span style="color:#1e3a8a">Next: Agentic AI II</span>

Everything above was a hand-built toy, so the loop itself would be visible. Next block, we hand the
"Thought" step to an actual agent (a real coding/research agent) and watch it work through a small
data task live &mdash; including a look at what it asks permission for, and why.